# Phase 5 — Modern Data Processing with Polars

**Goal:** Replicate the Phase 1 cleaning pipeline using Polars instead of pandas. Learn Polars' expression-based API and compare its syntax and performance to pandas.

**Libraries:** `polars`, `pandas`, `time`

**Input:** `../star_dataset.csv` (raw, unclean)

**Install if needed:** `pip install polars`

In [3]:
import polars as pl
import pandas as pd
import time

In [4]:
df = pl.read_csv("../star_dataset.csv")

print("Shape:", df.shape)
print("\nSchema:")
print(df.schema)
print()
print(df.head())

Shape: (1000, 6)

Schema:
Schema({'Name': String, 'Distance (ly)': Float64, 'Luminosity (L/Lo)': Float64, 'Radius (R/Ro)': Float64, 'Temperature (K)': Float64, 'Spectral Class': String})

shape: (5, 6)
┌───────────┬───────────────┬──────────────────┬───────────────┬─────────────────┬─────────────────┐
│ Name      ┆ Distance (ly) ┆ Luminosity       ┆ Radius (R/Ro) ┆ Temperature (K) ┆ Spectral Class  │
│ ---       ┆ ---           ┆ (L/Lo)           ┆ ---           ┆ ---             ┆ ---             │
│ str       ┆ f64           ┆ ---              ┆ f64           ┆ f64             ┆ str             │
│           ┆               ┆ f64              ┆               ┆                 ┆                 │
╞═══════════╪═══════════════╪══════════════════╪═══════════════╪═════════════════╪═════════════════╡
│ Altair    ┆ 16.594171     ┆ 9.979192         ┆ 1.63265       ┆ 7509.294247     ┆ A7V             │
│ Deneb     ┆ 2600.490723   ┆ 196002.627856    ┆ 202.970526    ┆ 8503.284796     ┆ A2Ia    

In [5]:
print("Rows before filter:", df.shape[0])

df = df.filter(pl.col("Luminosity (L/Lo)") >= 0)

print("Rows after filter: ", df.shape[0])
print(f"Removed {1000 - df.shape[0]} rows with negative luminosity")

Rows before filter: 1000
Rows after filter:  912
Removed 88 rows with negative luminosity


In [6]:
df_clean = (
    df
    .group_by("Name")
    .agg([
        pl.col("Distance (ly)").mean(),
        pl.col("Luminosity (L/Lo)").mean(),
        pl.col("Radius (R/Ro)").mean(),
        pl.col("Temperature (K)").mean(),
        pl.col("Spectral Class").first(),   # observations per star share the same class
    ])
    .sort("Name")
)

print("Unique stars:", df_clean.shape[0])
print()
print(df_clean)

Unique stars: 29

shape: (29, 6)
┌────────────┬───────────────┬──────────────┬───────────────┬───────────────────┬──────────────────┐
│ Name       ┆ Distance (ly) ┆ Luminosity   ┆ Radius (R/Ro) ┆ Temperature (K)   ┆ Spectral Class   │
│ ---        ┆ ---           ┆ (L/Lo)       ┆ ---           ┆ ---               ┆ ---              │
│ str        ┆ f64           ┆ ---          ┆ f64           ┆ f64               ┆ str              │
│            ┆               ┆ f64          ┆               ┆                   ┆                  │
╞════════════╪═══════════════╪══════════════╪═══════════════╪═══════════════════╪══════════════════╡
│ Achernar   ┆ 144.078936    ┆ 3149.761447  ┆ 9.209602      ┆ 15003.610593      ┆ B6Vep            │
│ Acrux      ┆ 320.916707    ┆ 25000.665022 ┆ 8.391405      ┆ 28001.16663       ┆ B0.5IV           │
│ Aldebaran  ┆ 65.028559     ┆ 517.761826   ┆ 44.179148     ┆ 3923.614977       ┆ K5III            │
│ Alnilam    ┆ 2000.090906   ┆ 53699.999016 ┆ 32.395467   

In [7]:
# Lazy API — Polars builds a query plan and optimises it before running anything
df_lazy = pl.scan_csv("../star_dataset.csv")

query = (
    df_lazy
    .filter(pl.col("Luminosity (L/Lo)") >= 0)
    .group_by("Name")
    .agg([
        pl.col("Distance (ly)").mean(),
        pl.col("Luminosity (L/Lo)").mean(),
        pl.col("Radius (R/Ro)").mean(),
        pl.col("Temperature (K)").mean(),
        pl.col("Spectral Class").first(),
    ])
    .sort("Name")
)

print("=== Optimised query plan ===")
print(query.explain())

result = query.collect()
print(f"\nResult shape: {result.shape}")
print(result.head())

=== Optimised query plan ===
SORT BY [col("Name")]
  AGGREGATE[maintain_order: false]
    [col("Distance (ly)").mean(), col("Luminosity (L/Lo)").mean(), col("Radius (R/Ro)").mean(), col("Temperature (K)").mean(), col("Spectral Class").first()] BY [col("Name")]
    FROM
    Csv SCAN [../star_dataset.csv]
    PROJECT */6 COLUMNS
    SELECTION: [(col("Luminosity (L/Lo)")) >= (0.0)]
    ESTIMATED ROWS: 1033

Result shape: (29, 6)
shape: (5, 6)
┌────────────┬───────────────┬──────────────┬───────────────┬───────────────────┬──────────────────┐
│ Name       ┆ Distance (ly) ┆ Luminosity   ┆ Radius (R/Ro) ┆ Temperature (K)   ┆ Spectral Class   │
│ ---        ┆ ---           ┆ (L/Lo)       ┆ ---           ┆ ---               ┆ ---              │
│ str        ┆ f64           ┆ ---          ┆ f64           ┆ f64               ┆ str              │
│            ┆               ┆ f64          ┆               ┆                   ┆                  │
╞════════════╪═══════════════╪══════════════╪══════

In [8]:
RUNS = 20   # repeat each pipeline N times for a stable average

# --- Pandas pipeline ---
def pandas_pipeline():
    df_pd = pd.read_csv("../star_dataset.csv")
    df_pd = df_pd[df_pd["Luminosity (L/Lo)"] >= 0]
    df_pd = (
        df_pd.groupby("Name", as_index=False)
        .agg({
            "Distance (ly)":      "mean",
            "Luminosity (L/Lo)":  "mean",
            "Radius (R/Ro)":      "mean",
            "Temperature (K)":    "mean",
            "Spectral Class":     "first",
        })
        .sort_values("Name")
    )
    return df_pd

# --- Polars pipeline ---
def polars_pipeline():
    return (
        pl.read_csv("../star_dataset.csv")
        .filter(pl.col("Luminosity (L/Lo)") >= 0)
        .group_by("Name")
        .agg([
            pl.col("Distance (ly)").mean(),
            pl.col("Luminosity (L/Lo)").mean(),
            pl.col("Radius (R/Ro)").mean(),
            pl.col("Temperature (K)").mean(),
            pl.col("Spectral Class").first(),
        ])
        .sort("Name")
    )

start = time.perf_counter()
for _ in range(RUNS):
    pandas_pipeline()
pandas_ms = (time.perf_counter() - start) / RUNS * 1000

start = time.perf_counter()
for _ in range(RUNS):
    polars_pipeline()
polars_ms = (time.perf_counter() - start) / RUNS * 1000

print(f"Pandas avg:  {pandas_ms:.2f} ms")
print(f"Polars avg:  {polars_ms:.2f} ms")
print(f"Speedup:     {pandas_ms / polars_ms:.2f}x  (on a 1000-row file — difference grows with data size)")

Pandas avg:  16.44 ms
Polars avg:  3.86 ms
Speedup:     4.26x  (on a 1000-row file — difference grows with data size)


In [9]:
df_clean.write_csv("../star_dataset_clean_polars.csv")
print("Saved to ../star_dataset_clean_polars.csv")

# Verify it matches the pandas-produced clean file
df_pandas_clean  = pd.read_csv("../star_dataset_clean.csv").sort_values("Name").reset_index(drop=True)
df_polars_reload = pd.read_csv("../star_dataset_clean_polars.csv").sort_values("Name").reset_index(drop=True)

# Align column order for comparison
shared_cols = ["Name", "Distance (ly)", "Luminosity (L/Lo)", "Radius (R/Ro)", "Temperature (K)", "Spectral Class"]
df_pandas_clean  = df_pandas_clean[shared_cols]
df_polars_reload = df_polars_reload[shared_cols]

matches = df_pandas_clean["Name"].reset_index(drop=True).equals(df_polars_reload["Name"].reset_index(drop=True))
print("Star names match between pandas and polars output:", matches)
print("\nPolars output preview:")
print(df_polars_reload.head())

Saved to ../star_dataset_clean_polars.csv
Star names match between pandas and polars output: True

Polars output preview:
               Name  Distance (ly)  Luminosity (L/Lo)  Radius (R/Ro)  \
0          Achernar     144.078936        3149.761447       9.209602   
1             Acrux     320.916707       25000.665022       8.391405   
2         Aldebaran      65.028559         517.761826      44.179148   
3           Alnilam    2000.090906       53699.999016      32.395467   
4  Alpha Centauri B       4.360539           3.221930       0.885471   

   Temperature (K) Spectral Class  
0     15003.610593          B6Vep  
1     28001.166630         B0.5IV  
2      3923.614977          K5III  
3     27502.303666           B0Ia  
4      5260.399450            K1V  
